In [1]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
from shapely import geometry
from scipy import ndimage
from shapely import wkb
from xgboost import XGBClassifier
import json
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *

def binary_classification_metrics(
    df: pd.DataFrame,
    target_column: str,
    pred_column: str
):
    """
    Compute accuracy, precision (class 1), and recall (class 1)
    from a dataframe containing binary labels (0/1).

    Returns a dict with metrics.
    """

    y_true = df[target_column].astype(int)
    y_pred = df[pred_column].astype(int)

    # Confusion matrix components
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    # Metrics
    accuracy = (tp + tn) / len(df) if len(df) > 0 else 0.0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision_class_1": precision_1,
        "recall_class_1": recall_1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_FOLDER = "/app/data/datasets/debug/bush/models"
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.json')
METADATA_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1_metadata.json')
load_dotenv()

db_string = os.getenv('DB_STRING_PROD')
engine = create_engine(db_string)

In [17]:
gdf_forests_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')
gdf_waters_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/COURS_D_EAU.shp')
gdf_u_zone = gpd.read_file('/app/data/datasets/debug/bush/sources/N_ZONE_URBA_S_034.gpkg',driver='GPKG')
gdf_forests_zones = gdf_forests_zones[gdf_forests_zones.forest_type==1]

/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


In [18]:
gdf_u_zone

,partition,libelle,libelong,typezone,destdomi,nomfic,urlfic,insee,datappro,datvalid,idurba,nomcom,gid,geometry
0,DU_34008,U2,Zone urbaine d'habitat diffus et de constructi...,U,None,Zonage GPU - MC - Approuvee le: 2026-02-11,None,34008,2026-02-11,2010-05-17,34008_PLU_20260211,LES AIRES,30001.0,"MULTIPOLYGON (((706824.869 6275764.092, 706828..."
1,DU_34008,U2,Zone urbaine d'habitat diffus et de constructi...,U,None,Zonage GPU - MC - Approuvee le: 2026-02-11,None,34008,2026-02-11,2010-05-17,34008_PLU_20260211,LES AIRES,30002.0,"MULTIPOLYGON (((707440.623 6276083.335, 707454..."
2,DU_34008,U4,"Zone d'activité urbanisée ""Vigne Grande""",U,None,Zonage GPU - MC - Approuvee le: 2026-02-11,None,34008,2026-02-11,2010-01-14,34008_PLU_20260211,LES AIRES,30003.0,"MULTIPOLYGON (((708040.607 6275745.381, 708028..."
3,DU_34008,U2,Zone urbaine d'habitat diffus et de constructi...,U,None,Zonage GPU - MC - Approuvee le: 2026-02-11,None,34008,2026-02-11,2010-05-17,34008_PLU_20260211,LES AIRES,30004.0,"MULTIPOLYGON (((706796.857 6275742.351, 706807..."
4,DU_34008,U4,"Zone d'activité urbanisée ""Vigne Grande""",U,None,Zonage GPU - MC - Approuvee le: 2026-02-11,None,34008,2026-02-11,2026-02-11,34008_PLU_20260211,LES AIRES,30005.0,"MULTIPOLYGON (((706205.297 6275942.931, 706181..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15613,DU_34290,Ap,None,A,None,Zonage GPU - M - Approuvee le: 2026-03-09,34290_reglement_20260309.pdf#page=69,34290,2026-03-09,2026-03-09,34290_PLU_20260309,SAINT-VINCENT-DE-BARBEYRARGUES,44010.0,"MULTIPOLYGON (((770543.54 6290856.195, 770581...."
15614,DU_34290,AUa,None,AUc,None,Zonage GPU - M - Approuvee le: 2026-03-09,34290_reglement_20260309.pdf#page=49,34290,2026-03-09,2026-03-09,34290_PLU_20260309,SAINT-VINCENT-DE-BARBEYRARGUES,44011.0,"MULTIPOLYGON (((770865.856 6289589.851, 770858..."
15615,DU_34290,UC,None,U,None,Zonage GPU - M - Approuvee le: 2026-03-09,34290_reglement_20260309.pdf#page=28,34290,2026-03-09,2026-03-09,34290_PLU_20260309,SAINT-VINCENT-DE-BARBEYRARGUES,44012.0,"MULTIPOLYGON (((770843.37 6289932.145, 770840...."
15616,DU_34290,UA,None,U,None,Zonage GPU - M - Approuvee le: 2026-03-09,34290_reglement_20260309.pdf#page=11,34290,2026-03-09,2026-03-09,34290_PLU_20260309,SAINT-VINCENT-DE-BARBEYRARGUES,44013.0,"MULTIPOLYGON (((770854.12 6290074.095, 770868...."


In [3]:
gdf_zones_r = gpd.read_postgis("select * from detections.n_dfci_old50m_s_034_ilots", geom_col='geom',con=engine)
gdf_zones_r

,id_ilot,code_ilot,compte_communal,insee_com,geom
0,103918,340198V00297_0,340198V00297,34198,"MULTIPOLYGON (((777980.658 6273167.472, 777960..."
1,104534,340199L00953_0,340199L00953,34199,"MULTIPOLYGON (((734417.82 6262732.91, 734410.0..."
2,104606,340199M01453_0,340199M01453,34199,"MULTIPOLYGON (((733334.706 6262686.729, 733326..."
3,104637,340199N00167_0,340199N00167,34199,"MULTIPOLYGON (((733716.05 6262894.98, 733716.4..."
4,104738,340199S00687_0,340199S00687,34199,"MULTIPOLYGON (((734263.5 6262804.4, 734265.23 ..."
...,...,...,...,...,...
168534,105967,340202D00714_0,340202D00714,34202,"MULTIPOLYGON (((760870.94 6276603.97, 760873.0..."
168535,106032,340202F00387_0,340202F00387,34202,"MULTIPOLYGON (((761060.53 6276468.42, 761062.0..."
168536,7323,340015N00005_0,340015N00005,34015,"MULTIPOLYGON (((690956.04 6255317.9, 690953.39..."
168537,2195,340006C00083_0,340006C00083,34006,"MULTIPOLYGON (((683512.99 6248082.81, 683520.8..."


In [4]:
# reload model
xgb_model_reloaded = XGBClassifier()
xgb_model_reloaded.load_model(MODEL_SAVE_PATH)

# reload metadata
with open(METADATA_SAVE_PATH) as f:
    metadata = json.load(f)
xgb_model_reloaded.scale_pos_weight = metadata["scale_pos_weight"]
THRESHOLD = metadata["decision_threshold"]
metadata

{'decision_threshold': 0.4141414141414142,
 'metric_optimized': 'f1',
 'scale_pos_weight': 0.7524071526822559}

# 1. Analysis on testset : Boissière

In [5]:
# load : 
df_test = pd.read_parquet(os.path.join(CACHE_DIR,'test.parquet'))
y = df_test['target_control']
x = df_test.drop(columns=['target_control','sample_id'])

In [6]:
x_test_business_features = x[['has_contact_river_zone','has_contact_forest_zone','has_inhabited_building','has_building']]
x_test_ml_features = x.drop(columns=['has_contact_river_zone','has_contact_forest_zone'])

In [7]:
gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/labels/target_dol_zones_v1.gpkg',driver='GPKG')

gdf_datas.to_crs('EPSG:2154', inplace=True)
gdf_zone_datas = gdf_datas[gdf_datas.insee_com=='34035']
gdf_zone_datas.head(5)

/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry
878,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19730,340035+00001_0,340035+00001,34035,0.0,0.0,"MULTIPOLYGON (((752023.38 6285086.06, 752021.8..."
879,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19731,340035+00003_0,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751926.52 6285261.73, 751928.5..."
880,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19733,340035+00003_2,340035+00003,34035,1.0,1.0,"MULTIPOLYGON (((751889.05 6284995.31, 751908.5..."
881,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19734,340035+00003_3,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751407.76 6282780.51, 751405.1..."
882,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19735,340035+00003_4,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751515.53 6282876.27, 751514.0..."


In [8]:
y_proba = xgb_model_reloaded.predict_proba(x_test_ml_features)[:, 1]
 
gdf_zone_datas["proba_control"] = y_proba
gdf_zone_datas["pred_control"] = 0
gdf_zone_datas.loc[gdf_zone_datas["proba_control"] >= THRESHOLD-0.2,"pred_control"] = 1
gdf_zone_datas

/opt/conda/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/conda/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry,proba_control,pred_control
878,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19730,340035+00001_0,340035+00001,34035,0.0,0.0,"MULTIPOLYGON (((752023.38 6285086.06, 752021.8...",0.196524,0
879,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19731,340035+00003_0,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751926.52 6285261.73, 751928.5...",0.210524,0
880,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19733,340035+00003_2,340035+00003,34035,1.0,1.0,"MULTIPOLYGON (((751889.05 6284995.31, 751908.5...",0.303386,1
881,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19734,340035+00003_3,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751407.76 6282780.51, 751405.1...",0.143034,0
882,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19735,340035+00003_4,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751515.53 6282876.27, 751514.0...",0.933388,1
...,...,...,...,...,...,...,...,...,...,...
1414,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20283,340035W00009_0,340035W00009,34035,1.0,1.0,"MULTIPOLYGON (((752968.29 6283179.48, 752980.8...",0.897884,1
1415,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20284,340035Z00001_0,340035Z00001,34035,1.0,1.0,"MULTIPOLYGON (((751619.601 6284418.833, 751615...",0.974782,1
1416,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20285,340035Z00002_0,340035Z00002,34035,0.0,0.0,"MULTIPOLYGON (((752284.406 6285301.143, 752279...",0.390612,1
1417,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20286,340035Z00003_0,340035Z00003,34035,0.0,0.0,"MULTIPOLYGON (((751395.27 6283379.04, 751406 6...",0.657149,1


In [9]:
df_features_imp = pd.DataFrame(columns = ['features', 'importance'], data = np.asarray([x_test_ml_features.columns, xgb_model_reloaded.feature_importances_]).T)
df_features_imp.sort_values(by='importance',ascending=False).head(20)

,features,importance
24,z50_12_surf_gt_200_ratio,0.169972
6,z50_12_mean,0.082077
63,has_building,0.025857
48,z30_12_count_gt_100_ratio,0.024025
43,z50_13_nb_surf_sup1m2_200,0.022421
39,z30_12_nb_surf_sup1m2_200,0.022187
5,z30_14_mean,0.021332
40,z30_13_nb_surf_sup1m2_200,0.021195
30,z30_12_nb_surf_sup1m2_100,0.020998
53,z50_14_count_gt_100_ratio,0.020853


In [10]:
test_results_gdf = postprocess_pred_control(gdf_zone_datas, x_test_business_features)

In [11]:
binary_classification_metrics(test_results_gdf,'target_control','pred_control_pp')

{'accuracy': np.float64(0.7818853974121996),
 'precision_class_1': np.float64(0.8054298642533937),
 'recall_class_1': np.float64(0.7035573122529645),
 'tp': 178,
 'fp': 43,
 'fn': 75,
 'tn': 245}

In [12]:
binary_classification_metrics(test_results_gdf,'target_pv','pred_control_pp')

{'accuracy': np.float64(0.7744916820702403),
 'precision_class_1': np.float64(0.7420814479638009),
 'recall_class_1': np.float64(0.7161572052401747),
 'tp': 164,
 'fp': 57,
 'fn': 65,
 'tn': 255}

In [ ]:
test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_boissiere_34035_dol_zones_xgb_v1.gpkg",
    driver="GPKG"
)

# 2. analysis on unknown geozone : Puisserguier

In [ ]:
communes_test = [
    {'name':'puisserguier', 'geozone_code': 34225,'geozone_id': 297,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_69.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_70.tif','/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_83.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_84.tif']}
]
imgs_bounds = []
for comm in communes_test :
    for img_path in comm['result_segmentation_files']:
        with rasterio.open(img_path) as src :
            bbox = src.bounds
            bbox_polygon = geometry.box(*bbox)
            print(bbox_polygon)
            imgs_bounds.append([img_path, bbox_polygon])

gdf_img = gpd.GeoDataFrame(data= imgs_bounds, columns=['image_path','geometry'], geometry='geometry', crs='EPSG:2154')
gdf_img.to_crs('EPSG:2154',inplace=True)
gdf_img.drop_duplicates(subset='image_path',inplace=True)

In [ ]:
gdf_zone_data = gdf_zones_r[gdf_zones_r.insee_com.isin([str(x['geozone_code']) for x in communes_test])]
gdf_zone_data


In [ ]:
gdf_zone_data = gpd.sjoin(gdf_img,gdf_zone_data, how='right', predicate='intersects').drop(columns='index_left')
gdf_zone_data = gdf_zone_data[~gdf_zone_data.image_path.isna()]
#gdf_zone_data.rename(columns={'geom':'geometry'}, inplace=True)
gdf_zone_data

In [ ]:
x_ml_features, x_business_features =  preprocess_features(gdf_zone_data, gdf_forests_zones, gdf_waters_zones, gdf_u_zones, cache_dir = CACHE_DIR, debug=False)

In [ ]:
y_proba = xgb_model_reloaded.predict_proba(x_ml_features)[:, 1]

#y_pred_custom = (y_proba >= THRESHOLD)
gdf_zone_data["proba_control"] = y_proba
gdf_zone_data["pred_control"] = 0
gdf_zone_data.loc[gdf_zone_data["proba_control"] >= THRESHOLD,"pred_control"] = 1
gdf_zone_data


In [ ]:


test_results_gdf = postprocess_pred_control(gdf_zone_data, x_business_features)

test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_puisserguier_34225_dol_zones_xgb_v1.gpkg",
    driver="GPKG"
)